# Lecture 16: Constrained Optimization — Penalty Method

---

```{note}
Lectures 13–15 developed three algorithms for unconstrained minimization — Gradient Descent, Newton's Method, and Quasi-Newton's Method — all of which assume the feasible region is all of $\mathbb{R}^n$. Most transportation engineering problems are not like that: budgets limit spending, road capacities cap flows, fleet sizes are finite. This lecture introduces the Penalty Method, the conceptually simplest strategy for handling constraints: convert the constrained problem into a sequence of unconstrained problems by appending a penalty term that grows whenever a constraint is violated.
```

**Learning Objectives**

By the end of this notebook, you will be able to:
1. State the Karush–Kuhn–Tucker (KKT) conditions and identify what each condition requires of a constrained optimum.
2. Formulate the quadratic penalty function for an inequality-constrained NLP and derive the penalized stationarity conditions.
3. Trace the Penalty Method iteration by hand and explain why the constrained optimum is recovered only in the limit as the penalty parameter $\rho \to \infty$.

**Prerequisites**: NLP Principles (Lecture 12); Gradient Descent, Newton's Method, BFGS (Lectures 13–15); calculus (partial derivatives); basic linear algebra (solving 2×2 systems).

**Estimated time**: 90 minutes (including in-class exercises)

---

## Optimality Conditions — KKT

Consider the standard-form NLP:

$$\min_{x} \; f(x) \quad \text{subject to} \quad g_i(x) \leq 0, \quad i = 1, \ldots, m$$

where $f, g_i : \mathbb{R}^n \to \mathbb{R}$ are continuously differentiable. The **Lagrangian** is:

$$\mathcal{L}(x, \lambda) = f(x) + \sum_{i=1}^{m} \lambda_i \, g_i(x)$$

If $x^*$ is a local minimum and a constraint qualification holds, then there exist multipliers $\lambda^* \in \mathbb{R}^m$ such that the following **Karush–Kuhn–Tucker (KKT) conditions** are satisfied:

| Condition | Statement | Interpretation |
|-----------|-----------|----------------|
| **Stationarity** | $\nabla f(x^*) + \sum_{i=1}^{m} \lambda_i^* \nabla g_i(x^*) = \mathbf{0}$ | Gradient of objective balanced by gradients of active constraints |
| **Primal feasibility** | $g_i(x^*) \leq 0 \quad \forall\, i$ | Solution lies within the feasible region |
| **Dual feasibility** | $\lambda_i^* \geq 0 \quad \forall\, i$ | Multipliers are non-negative for inequality constraints |
| **Complementary slackness** | $\lambda_i^* \, g_i(x^*) = 0 \quad \forall\, i$ | A constraint is either active ($g_i = 0$) or its multiplier is zero ($\lambda_i = 0$) |

For a **convex** NLP (convex $f$, convex $g_i$), the KKT conditions are both necessary and sufficient for global optimality. All methods in Lectures 16–19 seek a point that satisfies these four conditions.

---

## Constrained Optimization — Framework

The key insight behind the Constrained Optimization Frameworks is simple: if violating a constraint is made sufficiently expensive, the optimizer will avoid violations on its own. Lectures 16–19 introduce four algorithms that compute a KKT point by transforming the constrained problem into a sequence of tractable subproblems.

| Method | Core Idea | Subproblem Type | Feasibility of Iterates | Convergence |
|--------|-----------|-----------------|------------------------|-------------|
| **Penalty Method** | Append a penalty for constraint violation | Unconstrained minimization | Iterates are infeasible | Exact KKT point recovered only in the limit; linear rate |
| **Barrier Method** | Append a barrier that blows up near constraint boundary | Unconstrained minimization | Iterates are strictly feasible | Exact KKT point recovered only in the limit; linear rate |
| **Interior-Point** | Solve KKT system directly with barrier modification; Newton steps inside feasible region | Equality-constrained Newton system | Iterates are strictly feasible | Exact KKT point recovered only in the limit; polynomial-time; superlinear rate |
| **Sequential Quadratic Programming** | Newton on KKT conditions; each step solves a linearized QP | Inequality-constrained QP | Iterates may be infeasible | Exact for QP with linear constraints; superlinear rate in general |

```{note}
The Penalty Method approaches a KKT point from outside the feasible region: its iterates are generally infeasible (violating $g_i \leq 0$) but converge to feasibility as $\rho \to \infty$. The multiplier $\lambda_i^*$ is recovered as $\rho \cdot \max(0, g_i(x^*(\rho)))$ in the limit.
```

---

## 1. Penalty Method

For the problem $\min f(x)$ subject to $g_i(x) \leq 0$, Penalty Method introduces a **quadratic penalty function** — $P(x, \rho) = f(x) + \frac{\rho}{2} \sum_{i=1}^{m} \left[\max(0,\, g_i(x))\right]^2$. Here, the **violation** — $\max(0, g_i(x))$ equals zero when constraint $i$ is satisfied and equals $g_i(x)$ when it is violated, while the **penalty parameter** — $\rho > 0$ controls how harshly these violations are penalized. 

In practice, Penalty Method starts with a small initial value of the penalty parameter — $\rho^{(0)} > 0$ and iteratively solves a sequence of unconstrained subproblems — $x^{(k)} = \arg\min(P(x^{(k)}, \rho^{(k)}))$ with progressively larger penalty parameter — $\rho^{(k+1)} = \beta \cdot \rho^{(k)}$, until the violation falls under a pre-defined tolerance level — $\max_i \max(0, g_i(x^{(k)})) < \varepsilon$. As $\rho^{(k)} \to \infty$, the unconstrained subproblem solution converges to the KKT point of the original constrained problem. Consequently, comparing the stationarity conditions of the unconstrained subproblem with that of the original constrained problem recovers the Lagrange multiplier for each active constraint in the limit — $\lambda_i^* = \lim_{\rho \to \infty} \rho \cdot \max(0,\, g_i(x^*))$. Despite its conceptual simplicity, the Penalty Method suffers from two practical limitations: (i) as $\rho$ grows large, the Hessian of $P$ becomes increasingly ill-conditioned, slowing the inner unconstrained solver, and (ii) the final solution is always slightly infeasible, with constraint violation only driven below $\varepsilon$ rather than eliminated exactly.

---

## In-Class Exercises

### Exercise 1

MTC Chennai is optimizing a demand-responsive feeder service on the Tambaram–Chromepet corridor. The daily operating cost depends on fleet size $n$ (vehicles) and scheduled headway $h$ (minutes):

$$C(n, h) = (n - 5)^2 + 2(h - 3)^2$$

The depot has a combined resource constraint — vehicles and crew shifts — that limits $n + h \leq 7$.

1. Solve the unconstrained problem. Is the unconstrained optimum feasible?
2. Formulate the quadratic penalty function $P(n, h, \rho)$ for $\rho = 1$.
3. Solve the penalized subproblem analytically by setting $\nabla P = \mathbf{0}$. (Hint: you will get a $2 \times 2$ linear system.)
4. Fill in the penalty sequence table below for $\rho \in \{1, 5, 25, 125\}$ using the formula derived in part 3. What do you observe as $\rho$ increases?
5. Verify the KKT conditions at the constrained optimum $\left(n^* = \tfrac{13}{3},\, h^* = \tfrac{8}{3}\right)$ and interpret the multiplier $\lambda^*$.

#### Penalty Sequence Table

| $\rho$ | $n^*$ | $h^*$ | $n^* + h^*$ | $\max(0, n+h-7)$ | $\hat{\lambda}$ |
|--------|------------|------------|-------------|---------------------------|------------------------------------------------------|
| 1 | | | | | |
| 5 | | | | | |
| 25 | | | | | |
| 125 | | | | | |
| $\infty$ (KKT) | $13/3$ | $8/3$ | $7$ | $0$ | $4/3$ |

### Exercise 2

An NHAI corridor study models total system travel time across two parallel links — the Mumbai–Pune Expressway ($v_1$, thousand veh/hr) and the old NH48 ($v_2$, thousand veh/hr):

$$T(v_1, v_2) = 3v_1^2 - 2v_1 v_2 + 2v_2^2 - 14v_1 - 12v_2 + 50$$

A combined capacity constraint limits total flow: $v_1 + v_2 \leq 5$.

1. Solve the unconstrained problem. Verify the Hessian is positive definite. Is the unconstrained optimum feasible?
2. Formulate the quadratic penalty function $P(v_1, v_2, \rho)$.
3. Derive the penalized stationarity conditions $\nabla P = \mathbf{0}$ and show that the penalty subproblem solution satisfies:

$$v_1^*(\rho) = \frac{40 - 3\rho t(\rho)}{10}, \quad v_2^*(\rho) = \frac{50 - 4\rho t(\rho)}{10}, \quad t(\rho) = \frac{40}{10 + 7\rho}$$

4. Verify the KKT conditions at the constrained optimum $\left(v_1^* = \tfrac{16}{7},\, v_2^* = \tfrac{19}{7}\right)$ and interpret $\lambda^* = \tfrac{40}{7}$ in transportation terms.

---

## Take-Away Exercises

### Exercise 1

Delhivery is minimizing the daily operating cost at its Chennai cross-dock facility. The cost depends on daily throughput $\lambda$ (tonnes/day) and staffing level $s$ (workers):

$$C(\lambda, s) = 2\lambda^2 - 4\lambda s + 3s^2 - 12\lambda - 6s + 50$$

A combined budget and shift constraint limits $\lambda + 2s \leq 14$.

1. Solve the unconstrained problem analytically. Verify the Hessian is positive definite. Is the unconstrained optimum feasible?
2. Formulate the quadratic penalty function $P(\lambda, s, \rho)$ and derive the stationarity conditions $\nabla P = \mathbf{0}$ when the constraint is violated.
3. Solve the penalized subproblem analytically and compute $\lambda^*$ and $s^*$ as explicit functions of $\rho$. Fill in the table below.
4. Verify the KKT conditions at the true constrained optimum $\left(\lambda^* = \tfrac{116}{19},\, s^* = \tfrac{75}{19}\right)$ with $\lambda^* = \tfrac{64}{19}$. Interpret the multiplier: what does $\lambda^* > 0$ tell the Delhivery operations manager?

#### Penalty Sequence Table

| $\rho$ | $\lambda^*$ | $s^*$ | $\lambda^* + 2s^*$ | $\lambda + 2s - 14$ | $\hat{\lambda}$ |
|--------|-----------------|------------|-------------------|-----------|-------------------------|
| 1 | | | | | |
| 5 | | | | | |
| 25 | | | | | |
| 125 | | | | | |
| $\infty$ (KKT) | $116/19$ | $75/19$ | $14$ | $0$ | $64/19$ |

### Exercise 2

BMTC is optimizing its demand-responsive feeder service in Bengaluru. The total daily operating cost depends on fleet size $n$ (vehicles) and headway $h$ (minutes):

$$C(n, h) = 2n^2 + 3h^2 + 2nh - 20n - 30h + 100$$

A depot capacity constraint limits $n \leq 2$.

1. Solve the unconstrained problem analytically. Is the unconstrained optimum feasible with respect to $n \leq 2$?
2. Implement the Penalty Method in Python using `scipy.optimize.minimize` with `method='BFGS'` to solve the penalized subproblem at each outer iteration. Use $\rho_0 = 1$, scaling factor $\beta = 10$, and tolerance $\varepsilon = 10^{-4}$ on constraint violation. Start from $(n, h) = (0, 0)$.
3. Report $n^*(\rho)$, $h^*(\rho)$, the constraint violation, and the implicit multiplier $\hat{\lambda} = \rho \cdot \max(0, n^*(\rho) - 2)$ at each outer iteration. Compare your final solution to the KKT point $\left(n^* = 2,\, h^* = \tfrac{13}{3},\, \lambda^* = \tfrac{10}{3}\right)$.
4. Interpret $\lambda^* = \tfrac{10}{3}$ for the BMTC operations manager: what is the cost of tightening the depot capacity constraint by one vehicle?

---

## Further Reading

- Nocedal, J. and Wright, S.J. (2006). *Numerical Optimization* (2nd ed.). Springer — Chapter 17 (penalty and augmented Lagrangian methods).
- Bazaraa, M.S., Sherali, H.D., and Shetty, C.M. (2006). *Nonlinear Programming: Theory and Algorithms* (3rd ed.). Wiley — Chapter 12 (penalty function methods).
- Luenberger, D.G. and Ye, Y. (2016). *Linear and Nonlinear Programming* (4th ed.). Springer — Chapter 13 (penalty and barrier methods).
- SciPy documentation: `scipy.optimize.minimize` — [docs.scipy.org](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)